In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

1. Pivot bronze table so that years are rows and countries are columns

In [0]:
from pyspark.sql import functions as F

# 1. Read the new long-format Bronze table
bronze_df = spark.read.table("bronze.gdp_macro_raw")

# 2. Clean and standardize columns (rename 'economy' to 'Country_Code')
cleaned_long_df = (
    bronze_df
    .withColumnRenamed("economy", "Country_Code")
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("GDP_Growth_Pct", F.col("GDP_Growth_Pct").cast("decimal(18,2)"))
)

# 3. Pivot so that Country Codes become columns and Year remains the row index
gdp_final_spark = (
    cleaned_long_df
    .groupBy("Year")
    .pivot("Country_Code", ["USA", "EMU", "CHN", "JPN", "IND"])
    .agg(F.first("GDP_Growth_Pct"))
    .orderBy("Year")
)

# 4. Write to the Silver layer Delta table (overwriting or merging incrementally)
(
    gdp_final_spark.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.gdp_macro_cleaned")
)

print("Silver layer gdp_macro_cleaned successfully updated with the new format!")

2. Enforce datatypes

In [0]:
cleaned_silver_df = (
    gdp_final_spark
    .withColumn("Year", F.col("Year").cast(IntegerType()))
    .withColumn("USA", F.col("USA").cast(FloatType()))
    .withColumn("EMU", F.col("EMU").cast(FloatType()))
    .withColumn("CHN", F.col("CHN").cast(FloatType()))
    .withColumn("JPN", F.col("JPN").cast(FloatType()))
    .withColumn("IND", F.col("IND").cast(FloatType()))
)

3. Null check

In [0]:
# Check for any null counts across countries

null_check = cleaned_silver_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cleaned_silver_df.columns]
)

4. Write table to silver

In [0]:
(
    cleaned_silver_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.gdp_macro_cleaned")
)

print("Silver layer macro table successfully written!")